In [4]:
#!pip install spacy[transformers] stix2 fastapi uvicorn

In [5]:
#!python -m spacy download en_core_web_trf

In [1]:
import json
import ipaddress
import spacy
from spacy.language import Language
from spacy.tokens import Span

In [4]:
# GLOBAL VAR
path_threat_actors = r"C:\Users\Agee\OneDrive\Bureau\Work_dir\Filigran\cti_projects\Cyber_Threat_Intelligence\data\threat_actors.json"
path_report = r"C:\Users\Agee\OneDrive\Bureau\Work_dir\Filigran\cti_projects\Cyber_Threat_Intelligence\data\cleaned_text_pypdf.txt"

In [3]:
#############################################
# Chargement du modèle
#############################################

print("Chargement du modèle Transformer...")
nlp = spacy.load("en_core_web_trf")

#############################################
# EntityRuler
#############################################

ruler = nlp.add_pipe(
    "entity_ruler",
    before="ner",
    config={"overwrite_ents": False}
)

patterns = [

    ########################################################
    # CVE
    ########################################################

    {
        "label": "VULNERABILITY",
        "pattern": [
            {
                "TEXT": {
                    "REGEX": r"(?i)^CVE-\d{4}-\d{4,7}$"
                }
            }
        ]
    },

    ########################################################
    # SHA256
    ########################################################

    {
        "label": "SHA256",
        "pattern": [
            {
                "TEXT": {
                    "REGEX": r"^[A-Fa-f0-9]{64}$"
                }
            }
        ]
    },

    ########################################################
    # SHA1
    ########################################################

    {
        "label": "SHA1",
        "pattern": [
            {
                "TEXT": {
                    "REGEX": r"^[A-Fa-f0-9]{40}$"
                }
            }
        ]
    },

    ########################################################
    # MD5
    ########################################################

    {
        "label": "MD5",
        "pattern": [
            {
                "TEXT": {
                    "REGEX": r"^[A-Fa-f0-9]{32}$"
                }
            }
        ]
    },

    ########################################################
    # Email
    ########################################################

    {
        "label": "EMAIL",
        "pattern": [
            {
                "TEXT": {
                    "REGEX": r"^[^@\s]+@[^@\s]+\.[^@\s]+$"
                }
            }
        ]
    },

    ########################################################
    # Domain
    ########################################################

    {
        "label": "DOMAIN",
        "pattern": [
            {
                "TEXT": {
                    "REGEX": r"^(?:[a-zA-Z0-9-]+\.)+[A-Za-z]{2,}$"
                }
            }
        ]
    },

    ########################################################
    # URL
    ########################################################

    {
        "label": "URL",
        "pattern": [
            {
                "TEXT": {
                    "REGEX": r"^(?:https?|hxxps?)://.*"
                }
            }
        ]
    }
]

#############################################
# Threat Actors depuis un JSON
#############################################

try:

    with open(path_threat_actors, encoding="utf8") as f:
        actors = json.load(f)

    for actor in actors:

        tokens = actor.lower().split()

        patterns.append(
            {
                "label": "THREAT_ACTOR",
                "pattern": [{"LOWER": t} for t in tokens]
            }
        )

except FileNotFoundError:
    print("actors.json non trouvé.")

ruler.add_patterns(patterns)

#############################################
# Validation IPv4 / IPv6
#############################################

@Language.component("ip_detector")
def ip_detector(doc):

    new_ents = list(doc.ents)

    for token in doc:

        try:

            ip = ipaddress.ip_address(token.text)

            label = "IPv4" if ip.version == 4 else "IPv6"

            span = Span(doc, token.i, token.i + 1, label=label)

            new_ents.append(span)

        except ValueError:
            pass

    doc.ents = spacy.util.filter_spans(new_ents)

    return doc

nlp.add_pipe("ip_detector", after="entity_ruler")

#############################################
# Normalisation des CVE
#############################################

@Language.component("normalize_cve")
def normalize_cve(doc):

    for ent in doc.ents:

        if ent.label_ == "VULNERABILITY":
            ent._.set("normalized", ent.text.upper())

    return doc

if not Span.has_extension("normalized"):
    Span.set_extension("normalized", default=None)

nlp.add_pipe("normalize_cve", last=True)

print("Pipeline CTI chargé.")

Chargement du modèle Transformer...
Pipeline CTI chargé.


In [5]:
with open(path_report, encoding="utf8") as f:
    text = f.read()
print(text)

Date: April 29, 2025 Version: 1 Number of pages: 7 TARGETING AND COMPROMISE OF FRENCH ENTITIES USING THE APT28 INTRUSION SET ACTIVITIES ASSOCIATED WITH APT28 SINCE 2021 1Context ANSSI, alongside its partners at the Cyber Crisis Coordination Centre (C4), has observed the targeting and compromise of French entities by the APT28 intrusion set. Since 2021, this intrusion set has been used to gather strategic intelligence from entities located in France, Europe, Ukraine, and North America. In the context of the war of aggression started by Russia against Ukraine on the 24th of February 2022, espionage campaigns associated with the APT28 intrusion set and targeting Ukraine, North Atlantic Treaty Organisation countries, or European Union member states have been observed. The APT28 intrusion set Active since at least 2004, the APT28 intrusion set 1 is publicly attributed to the Russian Federation . This intrusion set is regularly used to target military and governmental organisations, as well 

In [6]:
doc = nlp(text)

In [7]:
print(f"Nombre d'entités : {len(doc.ents)}")

Nombre d'entités : 154


In [8]:
for ent in doc.ents[:30]:
    print(ent.text, "->", ent.label_)

April 29, 2025 -> DATE
1 -> CARDINAL
7 -> CARDINAL
APT28 -> THREAT_ACTOR
APT28 -> THREAT_ACTOR
ANSSI -> ORG
the Cyber Crisis Coordination Centre -> ORG
C4 -> ORG
French -> NORP
APT28 -> THREAT_ACTOR
2021 -> DATE
France -> GPE
Europe -> LOC
Ukraine -> GPE
North America -> LOC
Russia -> GPE
Ukraine -> GPE
the 24th of February 2022 -> DATE
APT28 -> THREAT_ACTOR
Ukraine -> GPE
North Atlantic Treaty Organisation -> ORG
European Union -> ORG
APT28 -> THREAT_ACTOR
at least 2004 -> DATE
APT28 -> THREAT_ACTOR
1 -> CARDINAL
the Russian Federation -> GPE
Europe -> LOC
the United States -> GPE
Russia -> GPE
